In [12]:
import pandas as pd
from pathlib import Path
import numpy as np
from scipy.signal import find_peaks
import os
os.listdir("../data/preprocessed/biosignals")

['Person10_D1_2_ID_2.csv',
 'Person11_D1_2_ID_3.csv',
 'Person12_D1_2_ID_4.csv',
 'Person13_D1_2_ID_5.csv',
 'Person14_D1_2_ID_6.csv',
 'Person15_D1_3_ID_1.csv',
 'Person16_D1_3_ID_2.csv',
 'Person17_D1_3_ID_3.csv',
 'Person18_D1_3_ID_4.csv',
 'Person19_D1_4_ID_1.csv',
 'Person1_D1_1_ID_1.csv',
 'Person20_D1_4_ID_2.csv',
 'Person21_D1_4_ID_3.csv',
 'Person22_D1_4_ID_4.csv',
 'Person23_D1_5_ID_1.csv',
 'Person24_D1_5_ID_2.csv',
 'Person25_D1_6_ID_1.csv',
 'Person26_D1_6_ID_2.csv',
 'Person2_D1_1_ID_2.csv',
 'Person3_D1_1_ID_3.csv',
 'Person4_D1_1_ID_4.csv',
 'Person5_D1_1_ID_5.csv',
 'Person6_D1_1_ID_6.csv',
 'Person7_D1_1_ID_7.csv',
 'Person8_D1_1_ID_8.csv',
 'Person9_D1_2_ID_1.csv']

## Compute the mean and std of the EDA for 30s chunks

In [ ]:
# Setup paths based on your group's directory structure
REPO_ROOT = Path(r"C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis")
# REPO_ROOT = Path(__file__).resolve().parent.parent
BIOSIGNAL_DIR = REPO_ROOT / "data" / "preprocessed" / "biosignals"
FEATURE_DIR = REPO_ROOT / "data" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = '30s'

# helper function
def temp_slope(series):
    """Calculates the linear slope of a 30-second window."""
    series = series.dropna()
    if len(series) < 2:
        return np.nan
    x = np.arange(len(series))
    # np.polyfit returns [slope, intercept], we just want slope.
    return np.polyfit(x, series.values, 1)[0]

# helper function
def eda_peaks(series):
    """Counts the number of peaks in the EDA signal for a 30-second window."""
    series = series.dropna()
    peaks, _ = find_peaks(series.values)
    return len(peaks)

def extract_features(file_path: Path) -> pd.DataFrame:
    """Reads a person's preprocessed biosignal file and extracts 30s window features."""
    
    # 1. Load data and set the datetime index
    df = pd.read_csv(file_path)
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')
    
    # Extract the person ID from the filename (e.g., "Person1_D1_1_1234")
    person_id = file_path.stem
    
    # 2. Group by round and phase to prevent "bleeding" across experimental boundaries
    # Then resample the time index into 30-second tumbling windows
    windowed = df.groupby(['round', 'phase']).resample(WINDOW_SIZE)
    
    # 3. Calculate statistics for each window
    # We start with just EDA mean and standard deviation
    features = windowed.agg({
        'HR': ['mean', 'std', 'max', 'min'], # Add a helper function to get the frequency domain HR signal
        'EDA': ['mean', 'std', 'max', 'min', eda_peaks],
        'TEMP': [temp_slope]
    })
    
    # 4. Clean up the multi-level columns created by .agg()
    # This turns ('EDA', 'mean') into 'eda_mean'
    features.columns = [f"{col[0].lower()}_{col[1]}" for col in features.columns]
    
    # 5. Clean up the index
    # Resampling creates windows where there might be no data
    features = features.dropna()
    features = features.reset_index()
    
    # Add our subject identifier back in
    features.insert(0, 'subject_id', person_id)
    
    return features

def build_feature_dataset():
    print(f"Scanning for preprocessed biosignals in {BIOSIGNAL_DIR}...")
    
    all_features = []
    processed_count = 0
    
    for file_path in BIOSIGNAL_DIR.glob("*.csv"):
        person_features = extract_features(file_path)
        all_features.append(person_features)
        processed_count += 1
        print(f"Processed features for: {file_path.stem} ({len(person_features)} windows)")
        
    if not all_features:
        print("No CSV files found. Check your BIOSIGNAL_DIR path.")
        return
        
    # Combine all subjects into one master dataset
    final_dataset = pd.concat(all_features, ignore_index=True)
    
    # Save
    output_path = FEATURE_DIR / "biosignal_features_30s.csv"
    final_dataset.to_csv(output_path, index=False)
    
    print("\n--- Feature Extraction Complete ---")
    print(f"Total subjects processed: {processed_count}")
    print(f"Total 30-second windows generated: {len(final_dataset)}")
    print(f"Dataset saved to: {output_path}")

build_feature_dataset()

Scanning for preprocessed biosignals in C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis\data\preprocessed\biosignals...
Processed features for: Person10_D1_2_ID_2 (134 windows)
Processed features for: Person11_D1_2_ID_3 (132 windows)
Processed features for: Person12_D1_2_ID_4 (132 windows)
Processed features for: Person13_D1_2_ID_5 (134 windows)
Processed features for: Person14_D1_2_ID_6 (131 windows)
Processed features for: Person15_D1_3_ID_1 (130 windows)
Processed features for: Person16_D1_3_ID_2 (130 windows)
Processed features for: Person17_D1_3_ID_3 (129 windows)
Processed features for: Person18_D1_3_ID_4 (129 windows)
Processed features for: Person19_D1_4_ID_1 (134 windows)
Processed features for: Person1_D1_1_ID_1 (145 windows)
Processed features for: Person20_D1_4_ID_2 (134 windows)
Processed features for: Person21_D1_4_ID_3 (133 windows)
Processed features for: Person22_D1_4_ID_4 (134 windows)
Processed features for: 

## Normalizing each person's data by the mean and std of their baseline period

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Adjust if your paths differ
REPO_ROOT = Path(r"C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis")
FEATURE_DIR = REPO_ROOT / "data" / "features"

def normalize_by_baseline(input_csv: str = "biosignal_features_30s.csv", 
                          output_csv: str = "biosignal_features_30s_baseline_norm.csv",
                          baseline_phase: str = "phase1"):
    
    file_path = FEATURE_DIR / input_csv
    print(f"Loading unnormalized features from {file_path}...")
    df = pd.read_csv(file_path)
    
    metadata_cols = ['subject_id', 'time', 'round', 'phase']
    # Drop metadata and ensure we only grab numeric columns
    numeric_df = df.drop(columns=metadata_cols, errors='ignore').select_dtypes(include=[np.number])
    feature_cols = numeric_df.columns.tolist()
    
    df_normalized = df.copy()
    
    print(f"Normalizing {len(feature_cols)} features using '{baseline_phase}' as the baseline...")
    
    # Process each subject individually
    for subject, group in df.groupby('subject_id'):
        
        # 1. Isolate this specific subject's baseline data
        baseline_data = group[group['phase'] == baseline_phase][feature_cols]
        
        # If a subject somehow doesn't have phase1 data, skip or warn
        if baseline_data.empty:
            print(f"  Warning: No baseline ({baseline_phase}) data for {subject}. Skipping normalization.")
            continue
            
        # 2. Calculate baseline mean and standard deviation
        b_mean = baseline_data.mean()
        b_std = baseline_data.std()
        
        # Edge Case Defense: If a sensor flatlines, std becomes 0. 
        # Division by zero creates NaNs. We replace 0s with a tiny number.
        b_std = b_std.replace(0, 1e-8)
        
        # 3. Apply the transformation to ALL phases for this subject
        normalized_features = (group[feature_cols] - b_mean) / b_std
        
        # 4. Insert the normalized values back into our main dataframe
        df_normalized.loc[group.index, feature_cols] = normalized_features

    # Save the final dataset
    out_path = FEATURE_DIR / output_csv
    df_normalized.to_csv(out_path, index=False)
    print(f"Success! Normalized dataset saved to {out_path}")

normalize_by_baseline()

Loading unnormalized features from C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis\data\features\biosignal_features_30s.csv...
Normalizing 10 features using 'phase1' as the baseline...
Success! Normalized dataset saved to C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis\data\features\biosignal_features_30s_baseline_norm.csv


C:\Users\Bruger\AppData\Local\Temp\ipykernel_29824\2966744499.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.78087703  0.53827151  0.40942993 -0.36361959 -0.75014434 -0.49246117
 -1.00782752 -0.75014434 -0.87898593  0.15174676 -0.87898593 -0.49246117
 -0.234778   -0.87898593 -0.75014434 -0.49246117  0.15174676 -0.62130276
  0.53827151 -1.00782752  0.15174676 -0.87898593 -2.29624338  0.02290517
 -0.62130276 -0.234778   -0.62130276  0.6671131  -0.234778    0.28058834
 -0.36361959  0.15174676 -0.49246117 -2.42508496  0.79595469  1.05363786
  0.40942993  0.79595469  0.28058834  1.05363786  0.92479627  0.92479627
  0.79595469  1.18247944 -2.0385602  -2.16740179  0.02290517  0.15174676
 -1.39435227 -0.75014434  0.15174676 -0.87898593 -0.87898593 -1.00782752
 -0.49246117  0.15174676 -2.16740179 -2.16740179  0.40942993 -0.87898593
 -0.49246117  0.53827151  0.02290517 -0.10593642  0.15174676 -0.36361959

## Also normalize the responses by the mean of the baseline period, so that we can compare across people

**NOTE**: We ignore the standard deviation since if not we would be dividing by zero in the case that a persons response is the same for phase1 across rounds.

In [ ]:
# Adjust if your paths differ
REPO_ROOT = Path(r"C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis")
RESPONSES_DIR = REPO_ROOT / "data" / "preprocessed" / "responses"
FEATURE_DIR = REPO_ROOT / "data" / "features"

def normalize_responses_by_baseline(output_csv: str = "responses_features_baseline_norm.csv",
                                    baseline_phase: str = "phase1"):
    
    print(f"Scanning for response files in {RESPONSES_DIR}...")
    
    # 1. Load and combine all individual response files
    all_responses = []
    for file_path in RESPONSES_DIR.glob("*.csv"):
        df = pd.read_csv(file_path)
        all_responses.append(df)
        
    if not all_responses:
        print("No CSV files found in the responses directory.")
        return
        
    combined_df = pd.concat(all_responses, ignore_index=True)
    
    # 2. Define metadata and features
    # 'difficulty' is explicitly excluded because it has no phase1 baseline
    metadata_cols = ['round', 'phase', 'participant_ID', 'puzzler', 'team_ID', 'E4_nr', 'difficulty']
    
    # Select only numeric columns for normalization, dropping metadata
    numeric_df = combined_df.drop(columns=metadata_cols, errors='ignore').select_dtypes(include=[np.number])
    feature_cols = numeric_df.columns.tolist()
    
    df_normalized = combined_df.copy()
    
    print(f"Normalizing {len(feature_cols)} emotion features using '{baseline_phase}' as the baseline...")
    
    # 3. Process each participant individually
    # Grouping by participant_ID to handle their specific psychological baseline
    for subject, group in combined_df.groupby('participant_ID'):
        
        # Isolate this specific subject's baseline data
        baseline_data = group[group['phase'] == baseline_phase][feature_cols]
        
        if baseline_data.empty:
            print(f"  Warning: No baseline data for participant {subject}. Skipping.")
            continue
            
        # Calculate baseline mean
        b_mean = baseline_data.mean()

        # We ignore the Standard Deviation to avoid the divide by zero error and since they should
        # all already be on the same scale
        
        # Apply the transformation to ALL phases for this subject
        normalized_features = group[feature_cols] - b_mean
        
        # Insert the normalized values back into our main dataframe
        df_normalized.loc[group.index, feature_cols] = normalized_features
        
    # Save the final dataset
    FEATURE_DIR.mkdir(parents=True, exist_ok=True)
    out_path = FEATURE_DIR / output_csv
    df_normalized.to_csv(out_path, index=False)
    
    print(f"Success! Normalized responses dataset saved to {out_path}")

normalize_responses_by_baseline()

Scanning for response files in C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis\data\preprocessed\responses...
Normalizing 11 emotion features using 'phase1' as the baseline...
Success! Normalized responses dataset saved to C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis\data\features\responses_features_baseline_norm.csv
